# Radia IH Notebook Interface

Source modules: `src/radia/ih_design.py` and `src/radia/ih_notebook.py`.

This notebook is the temporary comparison interface for induction-heating workflows. The same IHDesignSpec/headless contract is also operated through the Radia Simulink Induction Heating block; the intended migration direction is Simulink after the comparison gates pass.


In [1]:
# Version stamp for promoted panel notebooks
import json, platform, sys
try:
    import radia
    radia_version = getattr(radia, '__version__', 'unknown')
except Exception as exc:
    radia_version = f'not-importable: {exc}'
version_stamp = {
    'generated_at_utc': '2026-06-25T10:51:07Z',
    'runtime_python': sys.version.split()[0],
    'runtime_platform': platform.platform(),
    'runtime_radia_version': radia_version,
}
print(json.dumps(version_stamp, indent=2, ensure_ascii=False))


{
  "generated_at_utc": "2026-06-25T10:51:07Z",
  "runtime_python": "3.12.10",
  "runtime_platform": "Windows-2022Server-10.0.20348-SP0",
  "runtime_radia_version": "4.95.2"
}


## Migration Contract

- Source modules: `src/radia/ih_design.py` and `src/radia/ih_notebook.py`
- Migration state: `active-local-runner`
- Notebook target: `src/radia/panels/notebooks/radia_ih.ipynb`
- Rule: notebook code cells must not import desktop Qt bindings.
- Rule: solver work should remain in headless CLI/function modules.
- Rule: local runs save `run.log` and `result.json` under the notebook run root; `result.json` starts with `radia_result`.
- Rule: persistent initial values live in the notebook `DesignSpec(...)` cell; JSON files are run artifacts, not preset storage.


## Notebook Panel Notes

- This notebook is the IH comparison workbench: settings map to the app-specific `calc_*.py` CLI through `DesignSpec`, matching the Simulink block's headless solver contract.
- Edit the `DesignSpec(...)` cell to make new default values persistent; `result.json` and `run.log` are run artifacts.
- `Run local` executes in the background with timeout/cancel. Keep long-run outputs linked or saved under `runs/...`.
- Human visualization should use `netgen.webgui`; LLM/headless validation should use GMSH `.msh v4.1`; `.vol` and `.sol` stay as Netgen notebook IO.


## IH Workpiece Notes

- Workpiece impedance can be linear or nonlinear. Linear SIBC uses the material parameters and geometry to set one effective `Z_s` without a BH iteration.
- Nonlinear ESIM reads a two-column BH curve, solves a 1-D cell problem at the current operating field, and updates the effective surface impedance `Z_s` inside the Karl iteration.
- `esim_per_panel` trades speed for spatial detail by solving ESIM per boundary DOF/panel; scalar ESIM is the faster first check.
- Useful result keys: `impedance_model`, `esim_converged`, `esim_iterations`, `Z_s_wp_real`, `Z_s_wp_imag`, `P_wp_W`, and `esim_per_panel`.


## Previous Result Artifacts

The existing ESIM sweep data under `docs/ih_esim_benchmark/sweep_data_dense/` is a good reference before a new long run. Compare scalar and per-panel files such as `I100_f50k_scalar.json` and `I100_f50k_per_panel.json`; the README explains the reproduction flow and the dense sweep lives in `docs/ih_esim_benchmark/sweep_data_dense/`.


## Headless Backends

- `src/radia/ih_design.py`
- `src/radia/ih_notebook.py`
- `src/radia/panels/calc_inductance.py`
- `src/radia/panels/calc_fem_kelvin.py`
- `src/radia/panels/calc_heat.py`


In [2]:
from radia.ih_design import IHDesignSpec
from radia.ih_notebook import IHWorkbench

spec = IHDesignSpec()
workbench = IHWorkbench(spec)
workbench.display()


In [ ]:
# Build the current command after editing the widgets.
# The Run local button stays disabled until required inputs are present.
# Long runs execute in the background; use Cancel to terminate and `workbench.last_run` to inspect artifacts.
cmd = workbench.build_command()
cmd
